# Solutions · 15 · High-temperature thermochemistry: combustion and metals

Worked solutions to the exercises of [notebook 15](../notebooks/15_high_temperature_thermochemistry.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engthermo import combustion, reaction
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A gas turbine burns methane with 200 % excess air and reactants preheated to 700 K. Find the flame temperature, and the excess air needed to keep it at 1800 K (a typical turbine-inlet limit).

In [2]:
T_ad = combustion.adiabatic_flame_temperature("CH4", excess_air=2.0, T_in=700.0)["T_ad"]
e_needed = optimize.brentq(lambda e: combustion.adiabatic_flame_temperature("CH4", e, T_in=700.0)["T_ad"] - 1800.0, 0.0, 6.0)
print(f"200 % excess air, reactants at 700 K: flame temperature {T_ad:.0f} K; to hold 1800 K the excess air must be {e_needed:.0%}")

200 % excess air, reactants at 700 K: flame temperature 1480 K; to hold 1800 K the excess air must be 102%


Gas turbines burn lean - two to three times the stoichiometric air - precisely to hold the turbine-inlet
temperature within what the blade alloys and cooling can bear. The excess air is not wasted: it becomes
the working fluid that drives the turbine.

## Exercise 2

From the Ellingham diagram, at what temperature does the C/CO line cross the Si/SiO2 line? Compare with the operating temperature of a silicon smelting furnace (about 2000 K). Why is aluminium produced electrolytically instead?

In [3]:
f = lambda T: combustion.ellingham(T)["2 C + O2 -> 2 CO"][0] - combustion.ellingham(T)["Si + O2 -> SiO2"][0]
T_cross = optimize.brentq(f, 500, 3000)
print(f"carbon reduces silica above {T_cross:.0f} K ({T_cross-273.15:.0f} degC)")
f_al = lambda T: combustion.ellingham(T)["2 C + O2 -> 2 CO"][0] - combustion.ellingham(T)["4/3 Al + O2 -> 2/3 Al2O3"][0]
print(f"carbon would reduce alumina above {optimize.brentq(f_al, 500, 3500):.0f} K")

carbon reduces silica above 1935 K (1662 degC)
carbon would reduce alumina above 2309 K


Silicon smelting needs about 2000 K - exactly where the lines cross, which is why it is done in submerged-arc
furnaces at that temperature. Alumina would need well over 2000 K, where aluminium boils and carbides form;
the Hall-Heroult electrolytic process, which supplies the Gibbs energy electrically at 960 degC, is cheaper
than any furnace. Electrochemistry (notebook 16) is the alternative when the Ellingham diagram says no.

## Exercise 3

**Implement it yourself:** compute the CO2 emitted per MJ of heat for a coal approximated as pure carbon (LHV from the formation enthalpy of CO2), and compare with methane. What is the ratio?

In [4]:
lhv_C = -reaction.standard_enthalpy({"C": -1, "O2": -1, "CO2": 1})
co2_per_MJ_coal = 44.01e-3 / (lhv_C / 1e6) * 1e3
hv = combustion.heating_values("CH4")
co2_per_MJ_gas = 44.01e-3 / (hv["LHV"] / 1e6) * 1e3
print(f"carbon: LHV {lhv_C/1e3:.1f} kJ/mol -> {co2_per_MJ_coal:.0f} g CO2/MJ; methane: {co2_per_MJ_gas:.0f} g CO2/MJ; ratio {co2_per_MJ_coal/co2_per_MJ_gas:.2f}")

carbon: LHV 393.5 kJ/mol -> 112 g CO2/MJ; methane: 55 g CO2/MJ; ratio 2.04


Per unit of heat, burning carbon emits about twice the CO$_2$ of burning methane - the hydrogen in methane
carries almost half its heating value without any CO$_2$. Real coals contain some hydrogen and lie a little
below pure carbon, but the factor of nearly two is the thermodynamic core of coal-to-gas switching.